In [ ]:
## python modules used within this notebook
%matplotlib widget
import numpy as np
from scipy import integrate
from scipy import interpolate
import matplotlib.pyplot as plt
import matplotlib.animation
import matplotlib.colors as colors
import os
import h5py
import sys
import mynumerics as mn
import units
from IPython.display import display, Markdown
from IPython.display import HTML

import HHG
import MMA_administration as MMA

matplotlib.rcParams['animation.embed_limit'] = 200.

## TDSE with a custom input

We show the interface for the TDSE solver accessed directly through Python. We use this solver for a custom field we define, and then analyse the result in details. We will show the spectrum of the source term, wavefunction, we do energetic analyses via the Gabor transform and [invariant energetic distribution](https://doi.org/10.1103/PhysRevA.106.053115). Finally, we will show the depletion of the ground state.


First, we import the compiled dynamical library and its Pythonic wrapper:

In [ ]:
from PythonTDSE import *

# Compiled dynamic C library
path_to_DLL = os.path.join(os.environ['TDSE_1D_BUILD'],'libsingleTDSE_dev.so')
DLL = TDSE_DLL(path_to_DLL)

### Define the custom input field & numerical parameters

Here we define the input parameters for the CTDSE solver and the initial pulse. We show an example of a chirped pulse with a $\sin^2$-envelope. The field is then given by

$$ \mathcal{E}(t) = \mathcal{E}_0 \sin^2 \left( \frac{t}{T_{\text{envelope}}} \right) \cos \left(\omega_0 t + \omega_c t^2 \right) \,.$$

(Note that the instantaneous frequency is then $\omega_i(t) = \omega_0 + 2\omega_c t$. This means that $\omega_0$ cannot be taken as the central frequency, the frequency at the peak of the pulse is $\omega_i(\pi T_{\text{envelope}}/2) = \omega_0 + \pi \omega_c T_{\text{envelope}}$.)

In [ ]:
results_path = os.path.join(os.environ['MULTISCALE_WORK_DIR'],'mma_basics')
filename     = 'results.h5'

field_source = 'CTDSE' # CTDSE or CUPRAD

z_select_index = 0
r_select_index = 0

with h5py.File(os.path.join(results_path,filename),'r') as f:
    # retrieve the field eithr from CUPRAD or CTDSE
    if  (field_source=='CTDSE'):
        Efield_path = MMA.paths['CTDSE_outputs']+'/Efield' 
        E = f[Efield_path][z_select_index,r_select_index,:]
        tgrid_path  = MMA.paths['CTDSE_outputs']+'/tgrid' 
    elif (field_source=='CUPRAD'):
        Efield_path = MMA.paths['CUPRAD_outputs']+'/output_field' 
        E = f[Efield_path][z_select_index,:,r_select_index]
        tgrid_path  = MMA.paths['CUPRAD_outputs']+'/tgrid' 
    else:
        raise NameError('only CTDSE or CUPRAD can be sources of the field')
    tgrid = f[tgrid_path][:]

    # obtain default gas and set CTDSE potential accordingly
    try: gas_preset = f[MMA.paths['global_inputs']+'/gas_preset'][()].decode()
    except: raise ValueError('This tutorial works with preset gas options')
    Eguess = -HHG.Ip_list[gas_preset]
    trg_a  = HHG.soft_Coulomb_a[gas_preset]

    # list of numerical parameters
    dt    = f[MMA.paths['CTDSE_inputs']+'/dt'][()]
    dx    = f[MMA.paths['CTDSE_inputs']+'/dx'][()]
    num_r = 2*f[MMA.paths['CTDSE_inputs']+'/Nx_max'][()]
    CV    = f[MMA.paths['CTDSE_inputs']+'/CV_criterion_of_GS'][()]

    # rangoe for the volumetric integration representing ionisation
    x_int = f[MMA.paths['CTDSE_inputs']+'/x_int'][()]
    

    # get absorber parameters
    absorber = {'type' : 0} # no absorber by default
    if MMA.paths['CTDSE_inputs']+'/absorber_type' in f: # absorber is optional
        absorber_type = f[MMA.paths['CTDSE_inputs']+'/absorber_type'][()]
        absorber['type'] = absorber_type
        if (absorber_type == 1):
            absorber['x_cap'] =  f[MMA.paths['CTDSE_inputs']+'/absorber_x_cap'][()]
            absorber['alpha'] =  f[MMA.paths['CTDSE_inputs']+'/absorber_alpha'][()]
        elif (absorber_type == 2):
            absorber['x_cap'] =  f[MMA.paths['CTDSE_inputs']+'/absorber_x_cap'][()]
            


absorber ={'type'  : 1,
            'x_cap' : 50., # a.u.
            'alpha' : 0.001}

inputs_dict={'trg_a'    : trg_a,
             'Eguess'   : Eguess,
             'dt'       : dt,
             'dx'       : dx,
             'num_r'    : num_r, # 16000, #num_r,
             'CV'       : CV,
             'x_int'    : x_int,
             'absorber' : absorber, # {'type'  : 0}, # absorber,
             'writewft' : 1,
             'tprint'   : 1}


# omega0 = mn.ConvertPhoton(1000e-9,'lambdaSI','omegaau')
# chirp = 2e-4
# E_0 = 0.15   # peak electric field amplitude

# T0 = mn.ConvertPhoton(omega0,'omegaau','T0au') # the duration of the reference cycle
# T_max = 3*T0 # total pulse duration expressed in the number of the reference cycles
# N_t = 10000  # # of points for field construction (not for TDSE)

# # Construct the field
# tgrid = np.linspace(0, T_max, N_t)
# E = E_0* (np.sin(np.pi*tgrid/T_max)**2) *np.cos(omega0*tgrid + chirp*(tgrid)**2)


# # Create instance of input structure
# inputs = inputs_def()

# # Set the inputs for the TDSE solver
# trg_a = 1.1893 # Argon 
# inputs.init_default_inputs(
#             Eguess   = -0.5145 ,
#             trg_a    = trg_a ,     
#             dt       = 0.125 ,
#             dx       = 0.4 ,
#             num_r    = 16000 ,
#             writewft = 1 ,
#             tprint   = 1. ,
#             x_int    = 2. )
# # Note: Parameters currently needs to be fixed for the gauge-invariant energetic analysis (gas & some of numerics for the same ensemble of bound states)

### Pipeline to execute the TDSE computation

In [ ]:
inputs = inputs_def()
inputs.init_default_inputs(**inputs_dict)
inputs.init_time_and_field(DLL, E = E, t = tgrid) # set our electric field as the input
DLL.init_GS(inputs)                               # create the C-types input for the C-library
output = outputs_def()                            # prepare the structure that holds the TDSE outputs 
DLL.call1DTDSE(inputs, output)                    # run TDSE


# absorber ={'type'  : 1,
#             'x_cap' : 50., # a.u.
#             'alpha' : 0.001}

# absorber ={'type'  : 2,
#                                       'x_cap' : 20. # a.u.
#                                       },

### Obtain detailed analyses and visualisation
Here we specify some parameters for various analyses and plotting

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors


omega_max_plot = 3.5  # [a.u.]

# Optional spatial filtering of the wavefunction plot
filter_xgrid = False  # False: full x_grid; True: restrict the plotted range
x_max_plot = 250.1    # [a.u.], used only when filter_xgrid is True

# Optional lower-value filtering
filter_wavefunction_min = False
wavefunction_min = 1e-8
wavefunction_max = 0.5


# Load the wavefunction
t_psi, x_grid, wavefunction = output.get_wavefunction(
    inputs,
    grids=True
)


# -------------------------------------------------------------------------
# 1. Electric field
# -------------------------------------------------------------------------
fig1, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(
    output.get_tgrid(),
    output.get_Efield(),
    label='Electric field'
)

ax1.set_xlabel(r'$t~[\mathrm{a.u.}]$')
ax1.set_ylabel(r'$\mathcal{E}~[\mathrm{a.u.}]$')
ax1.legend()

fig1.tight_layout()
plt.show()


# -------------------------------------------------------------------------
# 2. Harmonic spectrum
# -------------------------------------------------------------------------
fig2, ax2 = plt.subplots(figsize=(8, 5))

ogrid = output.get_omegagrid()[:]
ko_max = mn.FindInterval(ogrid, omega_max_plot)

photon_energy = mn.ConvertPhoton(
    ogrid[:ko_max],
    'omegaau',
    'eV'
)

ax2.semilogy(
    photon_energy,
    np.abs(output.get_Fsourceterm())[:ko_max],
    label='Dipole acceleration spectrum'
)

ax2.set_xlim(photon_energy[[0, -1]])
ax2.set_xlabel(r'$\omega~[\mathrm{eV}]$')
ax2.set_ylabel(
    r'$|(\partial \hat{\jmath}/\partial t)(\omega)|'
    r'~[\mathrm{arb.~u.}]$'
)
ax2.legend()

fig2.tight_layout()
plt.show()


# -------------------------------------------------------------------------
# 3. Wavefunction
# -------------------------------------------------------------------------

# Figure size in inches: (width, height)
figsize_wavefunction = (8, 5.5)


# Select the spatial range
if filter_xgrid:
    x_range = np.abs(x_grid) < x_max_plot
else:
    x_range = slice(None)

x_plot = x_grid[x_range]


# Prepare the wavefunction data
psi_plot = np.abs(wavefunction).T[x_range]


# Optionally replace values below wavefunction_min
if filter_wavefunction_min:
    psi_display = np.maximum(
        psi_plot,
        wavefunction_min
    )
else:
    psi_display = psi_plot


# Logarithmic colour normalisation
wavefunction_norm = colors.LogNorm(
    vmin=wavefunction_min,
    vmax=wavefunction_max
)


fig3, ax3 = plt.subplots(
    figsize=figsize_wavefunction,
    layout='constrained'
)

pc3 = ax3.pcolormesh(
    t_psi,
    x_plot,
    psi_display,
    cmap='jet',
    norm=wavefunction_norm,
    shading='auto'
)

ax3.set_xlabel(r'$t~[\mathrm{a.u.}]$')
ax3.set_ylabel(r'$x~[\mathrm{a.u.}]$')

ax3.set_xlim(
    np.min(t_psi),
    np.max(t_psi)
)

ax3.set_ylim(
    np.min(x_plot),
    np.max(x_plot)
)


# Horizontal colour bar below the plot
cbar = fig3.colorbar(
    pc3,
    ax=ax3,
    orientation='horizontal',
    location='bottom',
    pad=0.08,
    shrink=0.8,
    aspect=45
)

cbar.set_label(r'$|\psi|~[\mathrm{a.u.}]$')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors


# -------------------------------------------------------------------------
# Gabor transform
# -------------------------------------------------------------------------

omega_max_plot = 3.5  # [a.u.]
Tmin_Gabor = 250.0     # [a.u.]
Tmax_Gabor = 1350.0    # [a.u.]


# Time-dependent source term
grad_V = output.get_sourceterm()

# Numerical parameters required by the Gabor transform
dt_Gabor = output.tgrid[1] - output.tgrid[0]
T = output.tgrid[output.Nt - 1]

# Calculate the Gabor transform
tgrid_Gabor, ogrid_Gabor, Gabor = DLL.gabor_transform(
    grad_V,
    dt_Gabor,
    output.Nt,
    omega_max_plot,
    Tmin_Gabor,
    Tmax_Gabor,
    1000,
    a=8
)


# -------------------------------------------------------------------------
# Plot the Gabor transform
# -------------------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8, 5.5),
    layout='constrained'
)

pc = ax.pcolormesh(
    tgrid_Gabor,
    mn.ConvertPhoton(
        ogrid_Gabor,
        'omegaau',
        'eV'
    ),
    Gabor,
    cmap='jet',
    norm=colors.LogNorm(
        vmin=1e-6,
        vmax=1.0
    ),
    shading='gouraud'
)

ax.set_xlabel(r'$t~[\mathrm{a.u.}]$')
ax.set_ylabel(r'Energy $[\mathrm{eV}]$')

cbar = fig.colorbar(
    pc,
    ax=ax
)

cbar.set_label(
    r'Gabor spectrogram $[\mathrm{arb.~u.}]$'
)

plt.show()

In [ ]:
inputs_dict={'trg_a'    : trg_a,
             'Eguess'   : Eguess,
             'dt'       : dt,
             'dx'       : dx,
             'num_r'    : 16000, #num_r,
             'CV'       : CV,
             'x_int'    : x_int,
             'absorber' : {'type'  : 0}, # absorber,
             'writewft' : 1,
             'tprint'   : 1}


inputs = inputs_def()
inputs.init_default_inputs(**inputs_dict)
inputs.init_time_and_field(DLL, E = E, t = tgrid) # set our electric field as the input
DLL.init_GS(inputs)                               # create the C-types input for the C-library
output2 = outputs_def()                            # prepare the structure that holds the TDSE outputs 
DLL.call1DTDSE(inputs, output)                    # run TDSE


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors


# -------------------------------------------------------------------------
# Gabor transform
# -------------------------------------------------------------------------

omega_max_plot = 3.5  # [a.u.]
Tmin_Gabor = 250.0     # [a.u.]
Tmax_Gabor = 1350.0    # [a.u.]


# Time-dependent source term
grad_V = output.get_sourceterm()

# Numerical parameters required by the Gabor transform
dt_Gabor = output.tgrid[1] - output.tgrid[0]
T = output.tgrid[output.Nt - 1]

# Calculate the Gabor transform
tgrid_Gabor, ogrid_Gabor, Gabor = DLL.gabor_transform(
    grad_V,
    dt_Gabor,
    output.Nt,
    omega_max_plot,
    Tmin_Gabor,
    Tmax_Gabor,
    1000,
    a=8
)


# -------------------------------------------------------------------------
# Plot the Gabor transform
# -------------------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8, 5.5),
    layout='constrained'
)

pc = ax.pcolormesh(
    tgrid_Gabor,
    mn.ConvertPhoton(
        ogrid_Gabor,
        'omegaau',
        'eV'
    ),
    Gabor,
    cmap='jet',
    norm=colors.LogNorm(
        vmin=1e-6,
        vmax=1.0
    ),
    shading='gouraud'
)

ax.set_xlabel(r'$t~[\mathrm{a.u.}]$')
ax.set_ylabel(r'Energy $[\mathrm{eV}]$')

cbar = fig.colorbar(
    pc,
    ax=ax
)

cbar.set_label(
    r'Gabor spectrogram $[\mathrm{arb.~u.}]$'
)

plt.show()